# T30 / E13 — Đổi mô hình đọc sang Sailor2-8B

**Câu hỏi của thí nghiệm này không phải "mô hình nào điểm cao hơn".**

Sailor2-8B-SFT mở rộng từ chính họ Qwen2.5 rồi huấn luyện thêm rất nhiều trên ngôn ngữ Đông Nam
Á. Nên nó là một phép thử tự nhiên: **vị trí các đầu chú ý có ích là thuộc tính của kiến trúc,
hay của dữ liệu huấn luyện?**

Nếu các đầu mà bộ dò tuyến tính dựa vào nằm ở cùng độ sâu trong cả hai mô hình, thì vị trí đầu
sao chép sống sót qua việc huấn luyện chuyên sâu một ngôn ngữ khác. Nếu chúng dịch chuyển thì
ngược lại. **Cả hai đều là kết quả** — kết quả thứ hai đừng lược đi.

Bài gốc Lookback Lens có một kết quả liên quan: bộ dò huấn luyện trên mô hình 7B dùng lại được
cho 13B mà không cần huấn luyện lại. Ô này kiểm điều tương tự theo hướng khác — không đổi cỡ mà
đổi dữ liệu huấn luyện.

## Khác T27 ở ba chỗ, và cả ba đều là chỗ dễ hỏng

1. **Lớp tràn số của Sailor2 chưa biết.** Với Qwen2.5-7B, T07 đo được đúng lớp 27 — lớp cuối —
   hỏng ở `float16` trên 20/20 mẫu. Sailor2 là mô hình khác, nhiều lớp hơn, nên lớp hỏng của nó
   là **câu hỏi thực nghiệm**. Ô 5 đo trước rồi tự ghi vào cấu hình; `exclude_layers` để trống
   trong repo là cố ý.
2. **Lưới lớp × đầu có thể khác.** Qwen cho 28 lớp; Sailor2 là bản mở rộng nên nhiều hơn. Phần
   so vị trí đầu xử lý riêng trường hợp này, xem ô 9.
3. **Mẫu prompt.** Chat template của Sailor2 có thể khác Qwen đôi chút. Không phải lo: từ T07,
   vị trí ngữ cảnh và phản hồi được tìm bằng cách **dò chuỗi trong prompt đã render** rồi ánh xạ
   sang token, chứ không đếm ký tự khung. Ô 6 in ra để đối chiếu.

## Chi phí

Khoảng **3 giờ card đồ họa** cho 7.000 mẫu ViHallu, cộng ~10 phút cho ô dò kiểu số. Nằm gọn
trong hạn mức 30 giờ/tuần.

**Chấm điểm chạy ở máy cá nhân**, theo quy tắc chốt ở T23: điểm dev lệch tới 0,0075 giữa hai môi
trường vì bộ giải tối ưu hội tụ khác nhau. Notebook này chỉ trích đặc trưng.


## Chuẩn bị

Ô 3 là ô tiền kiểm. Ô 4 chuẩn hóa và chia tập — **đừng bỏ**, đây đúng chỗ lượt chạy T27 hỏng
lần thứ hai.

In [ ]:
# Ô 1 — lấy code. Chạy lại được nhiều lần.
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/wsunicorn/vihallulens.git"
REPO_DIR = Path("/kaggle/working/vihallulens")


def run(*args, cwd=None):
    print("$", " ".join(str(a) for a in args))
    result = subprocess.run(args, cwd=cwd, capture_output=True, text=True)
    print(result.stdout.strip())
    if result.returncode:
        print(result.stderr.strip())
        raise SystemExit(f"lệnh hỏng: {' '.join(str(a) for a in args)}")
    return result.stdout


if REPO_DIR.exists():
    run("git", "fetch", "--all", cwd=REPO_DIR)
    run("git", "reset", "--hard", "origin/main", cwd=REPO_DIR)
else:
    run("git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR))

os.chdir(REPO_DIR)
run("git", "log", "-1", "--format=%h %s")

In [ ]:
# Ô 2 — cài đặt. bitsandbytes cần cho lượng tử hóa 4 bit.
#
# Sailor2-8B nặng hơn Qwen2.5-7B nên phần tải trọng số lâu hơn, khoảng 8-10 phút lần đầu.
!pip install -q --no-deps -e .
!pip install -q bitsandbytes

In [ ]:
# Ô 3 — TIỀN KIỂM. Vài giây, chạy trước mọi thứ.
#
# Cùng lối viết đã chốt ở T27: đi HẾT chuỗi phụ thuộc, tách rành mạch hai nhóm.
#
#   PHẢI CÓ SẴN  - phiên này không tạo được: dữ liệu thô đã mount, gói đã cài
#   TỰ TẠO       - các ô sau sinh ra theo thứ tự, liệt kê để đọc và đối chiếu
import importlib.util
import sys
from pathlib import Path

sys.path.insert(0, "src")
from vihallulens.config import extraction_hash, load_config
from vihallulens.data.paths import find_raw_dir

CFG = "configs/e13_sailor2_vihallu.yaml"
cfg = load_config(CFG)
problems = []

# --- PHẢI CÓ SẴN -------------------------------------------------------------------------------
packages = ("torch", "transformers", "bitsandbytes", "pandas", "accelerate")
absent = [name for name in packages if importlib.util.find_spec(name) is None]
trang_thai = f"THIEU {absent}" if absent else f"du ca {list(packages)}"
print(f"  goi phai co san   : {trang_thai}")
if absent:
    problems.append(f"thieu goi {absent}")

try:
    raw = find_raw_dir()
    files = sorted(p.name for p in Path(raw).glob("vihallu*"))
    print(f"  du lieu tho       : {raw}")
    print(f"  file vihallu tho  : {files or 'KHONG CO'}")
    if not files:
        problems.append("khong thay file vihallu nao trong du lieu tho")
except Exception as error:
    print(f"  du lieu tho       : KHONG TIM THAY ({error})")
    problems.append("chua mount dataset du lieu tho")

print(f"  mo hinh doc       : {cfg.extractor.model_name}")
print(f"  exclude_layers    : {cfg.extractor.exclude_layers}  <- o 5 se ghi de")

# --- TỰ TẠO, theo thứ tự các ô sau ---------------------------------------------------------------
chain = [
    ("o 4", "data/interim/vihallu_{train,dev,test}.parquet", "normalize_data + split_data"),
    ("o 5", "configs/e13_sailor2_vihallu.yaml co exclude_layers", "compare_dtypes"),
    ("o 7", "data/processed/vihallu_{split}_<hash>.jsonl", "extract_features"),
]
print("  chuoi tu tao:")
for cell, target, maker in chain:
    print(f"    {cell:<5} {target:<52} <- {maker}")

if problems:
    raise SystemExit("TIEN KIEM HONG: " + "; ".join(problems))
print("\nTien kiem dat: dieu kien ngoai da du, phan con lai phien nay tu tao.")

In [ ]:
# Ô 4 — chuẩn bị dữ liệu và kiểm môi trường. Khoảng 2 phút, CPU.
!python scripts/probe_env.py
!python scripts/normalize_data.py --dataset vihallu
!python scripts/split_data.py --only vihallu
!python -m pytest tests/test_compare_heads.py -q

## Dò lớp tràn số — bắt buộc, và không suy ra được từ Qwen

Ô 5 tốn khoảng 10 phút GPU và **quyết định cả lượt chạy 3 giờ phía sau**. Nó đo bằng cách chạy
song song `float16` và `float32` trên cùng vài mẫu rồi so từng lớp.

**Đọc gì:** danh sách lớp tràn số. Với Qwen là đúng một lớp, lớp cuối. Nếu Sailor2 ra nhiều lớp
hoặc ra lớp ở giữa thì dừng lại đọc kỹ — mất nhiều lớp giữa sẽ làm phép so với Qwen khập khiễng
và phải ghi rõ trong báo cáo.

Ô 5 tự ghi kết quả vào `configs/e13_sailor2_vihallu.yaml`. **Nhớ commit lại file đó** sau khi
chạy xong, nếu không lượt chạy này không tái lập được.

In [ ]:
# Ô 5 — DÒ LỚP TRÀN SỐ CỦA SAILOR2. Khoảng 10 phút GPU. BẮT BUỘC chạy trước ô 7.
#
# Đây là ô quan trọng nhất notebook này. Với Qwen2.5-7B, T07 đo được đúng lớp cuối tràn số ở
# float16 trên 20/20 mẫu, và bỏ nó đi thì 27 lớp còn lại khớp float32 tới 0,07 % thang đo.
#
# Sailor2 là mô hình KHÁC. Lớp hỏng của nó không suy ra được từ Qwen — phải đo. Bỏ qua ô này thì
# mọi đặc trưng trích ra sẽ có nan ở ít nhất một lớp, và điều đó chỉ lộ ra sau 3 giờ GPU.
import ast
import re
from pathlib import Path

proc = subprocess.run(
    ["python", "scripts/compare_dtypes.py", "--model", "sail/Sailor2-8B-SFT",
     "--per-dataset", "10"],
    capture_output=True, text=True,
)
print(proc.stdout[-7000:])
if proc.returncode:
    print(proc.stderr[-3000:])
    raise SystemExit("do kieu so hong")

# compare_dtypes.py in ra mot dong may doc duoc: EXCLUDE_LAYERS=[27]
found = re.search(r"^EXCLUDE_LAYERS=(\[.*\])$", proc.stdout, flags=re.MULTILINE)
if not found:
    raise SystemExit("khong thay dong EXCLUDE_LAYERS= trong output — doc ky stdout ben tren")
bad = sorted(set(ast.literal_eval(found.group(1))))
print()
print(f"  Lop tran so do duoc: {bad if bad else 'KHONG CO'}")

if not bad:
    raise SystemExit(
        "Khong lop nao tran so. Voi Qwen thi lop cuoi luon tran, nen ket qua nay dang ngo.\n"
        "Hai kha nang, va phai phan biet duoc truoc khi chay tiep:\n"
        "  - Sailor2 that su on o float16. Co the that, va la mot phat hien dang ghi.\n"
        "  - Phep do chua cham toi lop hong vi 10 mau qua it hoac ngu canh qua ngan.\n"
        "Chay lai voi --per-dataset 20 roi doc bang per-layer ben tren truoc khi quyet."
    )

# Ghi thang vao cau hinh de extraction_hash phan anh dung thu da chay. Ghi de an toan vi
# exclude_layers trong repo de trong co chu dich.
CFG_PATH = Path("configs/e13_sailor2_vihallu.yaml")
text = CFG_PATH.read_text(encoding="utf-8")
patched = re.sub(r"^  exclude_layers: \[\]$", f"  exclude_layers: {bad}", text, count=1,
                 flags=re.MULTILINE)
if patched == text:
    raise SystemExit("khong tim thay dong 'exclude_layers: []' de ghi de — kiem tra lai config")
CFG_PATH.write_text(patched, encoding="utf-8")
print(f"  Da ghi vao config : exclude_layers: {bad}")
print("  NHO commit lai config nay sau khi chay xong, de luot chay tai lap duoc.")

In [ ]:
# Ô 6 — cổng kiểm trước khi tiêu 3 giờ GPU. Vài giây, CPU.
import sys

sys.path.insert(0, "src")
from importlib import reload

from transformers import AutoTokenizer

import vihallulens.config as config_module
from vihallulens.extract.prompt import render_prompt

reload(config_module)
cfg = config_module.load_config("configs/e13_sailor2_vihallu.yaml")
run = config_module.extraction_hash(cfg)

print(f"  mo hinh doc       : {cfg.extractor.model_name}")
print(f"  exclude_layers    : {cfg.extractor.exclude_layers}")
print(f"  hash trich        : {run}")

if not cfg.extractor.exclude_layers:
    raise SystemExit("exclude_layers van trong — o 5 chua chay hoac chua ghi duoc. DUNG LAI.")

# Chat template cua Sailor2 co the khac Qwen. Tu T07, vi tri ngu canh va phan hoi duoc tim bang
# cach DO CHUOI trong prompt da render roi anh xa sang token, chu khong dem ky tu khung. O day
# kiem thang tinh chat do thay vi chi in ra nhin bang mat.
NGU_CANH = "Hà Nội là thủ đô của Việt Nam."
CAU_HOI = "Thủ đô Việt Nam là gì?"
PHAN_HOI = "Thủ đô là Hà Nội."

tok = AutoTokenizer.from_pretrained(cfg.extractor.model_name)
sample = render_prompt(tok, context=NGU_CANH, question=CAU_HOI, response=PHAN_HOI)

print()
print("  --- prompt da render ---")
print(sample.text)
print("  --- het ---")
print(f"  vung ngu canh doc lai : {sample.context!r}")
print(f"  vung phan hoi doc lai : {sample.response!r}")

assert sample.context == NGU_CANH, "vung ngu canh lech — DUNG LAI, dung trich dac trung"
assert sample.response == PHAN_HOI, "vung phan hoi lech — DUNG LAI, dung trich dac trung"
print()
print("  Hai vung khop chinh xac, nen chat template khac Qwen cung khong lam lech vi tri.")

## Trích đặc trưng

Khoảng **3 giờ**. Chạy lại được — phần đã xong không mất.

**Đọc gì trong lúc chạy:** dòng `lớp bỏ` phải khớp danh sách ô 5 đo được, và `lỗi` phải là 0.
Nếu thấy `nan` xuất hiện thì dừng ngay: nghĩa là còn lớp tràn số mà ô 5 chưa bắt được.

In [ ]:
# Ô 7 — trích đặc trưng. Khoảng 3 giờ. Chạy lại được, có lưu tiến độ.
#
# Ba tập chạy nối nhau. Nếu phiên đứt thì chạy lại ô này, phần đã xong không mất.
!python scripts/extract_features.py --config configs/e13_sailor2_vihallu.yaml --split train
!python scripts/extract_features.py --config configs/e13_sailor2_vihallu.yaml --split dev
!python scripts/extract_features.py --config configs/e13_sailor2_vihallu.yaml --split test

In [ ]:
# Ô 8 — kiểm toàn vẹn shard trước khi rời phiên. Vài giây, CPU.
import sys
from pathlib import Path

sys.path.insert(0, "src")
sys.path.insert(0, "scripts")
import numpy as np

from extract_features import load_done, shard_path
from vihallulens.config import load_config

cfg = load_config("configs/e13_sailor2_vihallu.yaml")
run = extraction_hash(cfg)
ok = True
for split in ("train", "dev", "test"):
    path = shard_path(Path("data/processed"), run, "vihallu", split)
    rows = list(load_done(path).values())
    if not rows:
        print(f"  {split:<6}: TRONG — {path}")
        ok = False
        continue
    values = np.asarray([r["lookback_total"] for r in rows], dtype=np.float32)
    finite = np.isfinite(values).all()
    in_range = ((values >= 0) & (values <= 1)).all()
    n_layers = len(rows[0]["layer_indices"])
    n_heads = values.shape[1] // n_layers
    print(f"  {split:<6}: {len(rows):>6,} mau | luoi {n_layers} x {n_heads} "
          f"| huu han {finite} | trong [0,1] {in_range}")
    ok = ok and finite and in_range

print()
print(f"  Luoi cua Sailor2: {n_layers} lop x {n_heads} dau = {n_layers * n_heads:,} cap")
print("  Qwen2.5-7B de so: 27 lop x 28 dau = 756 cap")
if (n_layers, n_heads) != (27, 28):
    print("  -> HAI LUOI KHAC NHAU. Phan so vi tri dau se chi bao do sau tuong doi, dung")
    print("     so theo chi so. Xem docstring scripts/compare_heads.py.")

if not ok:
    raise SystemExit("Shard co van de — dung dem ve, chay lai o 7.")
print("\nShard sach.")

## Chấm điểm — KHÔNG chạy ở đây

Theo quy tắc chốt ở T23, mọi phép so sánh phải chấm trên **cùng một máy**. Điểm dev lệch tới
0,0075 giữa Kaggle và máy cá nhân vì bộ giải tối ưu của hồi quy logistic hội tụ khác nhau giữa
hai phiên bản Python — mà biên độ đề tài đang xét chỉ cỡ 0,01.

Ô 9 gói shard lại để tải về. Chấm ở máy.

In [ ]:
# Ô 9 — lấy kết quả về. Vài giây.
#
# Tải ba file này về máy cá nhân rồi chạy CHẤM ĐIỂM Ở ĐÓ, theo quy tắc chốt ở T23.
import shutil
import sys
from pathlib import Path

sys.path.insert(0, "src")
from vihallulens.config import extraction_hash, load_config

cfg = load_config("configs/e13_sailor2_vihallu.yaml")
run = extraction_hash(cfg)

out = Path("/kaggle/working/ket_qua_t30")
out.mkdir(exist_ok=True)
for split in ("train", "dev", "test"):
    src = Path(f"data/processed/vihallu_{split}_{run}.jsonl")
    if src.exists():
        shutil.copy(src, out / src.name)
shutil.copy("configs/e13_sailor2_vihallu.yaml", out / "e13_sailor2_vihallu.yaml")

for f in sorted(out.iterdir()):
    print(f"  {f.name:<44} {f.stat().st_size / 1e6:>8.1f} MB")

print("""
Tai het thu muc ket_qua_t30 ve may, dat vao:
  *.jsonl  ->  data/processed/
  e13_sailor2_vihallu.yaml  ->  configs/   (GHI DE — no mang exclude_layers da do)

Roi chay o may ca nhan:
  python scripts/run_chunk_aware.py --config configs/e13_sailor2_vihallu.yaml
  python scripts/compare_heads.py --config-a configs/e03_chunk_sentence_vihallu.yaml \\
                                  --config-b configs/e13_sailor2_vihallu.yaml
""")